In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from constants import predicted_dir, predictor_vars, target_var, converted_dir
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split



In [2]:
df_test = pd.read_csv(f"{converted_dir}/vehiculos_test.csv")
df_train = pd.read_csv(f"{converted_dir}/vehiculos_train.csv")

## Implemente al menos 3 modelos de clasificación e intente superar al baseline compartido.
Para los 3 modelos utilice el mismo algoritmo de clasificación (elija uno aprendido en clases) con diferentes configuraciones. Distintas configuraciones pueden incluir distintos valores de hiperparámetros o técnicas de procesamiento de los datos de entrada. Para las distintas configuraciones puede considerar lo siguiente:
- Sus datos podrían ser estandarizados/normalizados o no.
- Puede definir alguna estrategia para codificar variables categóricas (puede utilizar One-hot-encoding, Target encoding, u otra estrategia).
- Identifique valores faltantes y defina una estrategia (de ser necesario) para la imputación de datos.
- Analice las variables categóricas y determine si es necesario tratar categorías poco frecuentes (por ejemplo, a veces agrupar categorías raras puede ser útil). Explique sus decisiones en el tratamiento de los datos.
- Detecte valores atípicos y defina una estrategia para tratarlos. Considere que eliminar los registros outliers (del set de entrenamiento) no es la única estrategia posible (puede considerar la regla del rango intercuartil, Local Outlier Factor, u otra estrategia). Explique sus decisiones en el tratamiento de los datos.



In [3]:
X = df_train[predictor_vars]
y = df_train[target_var]

In [4]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

In [5]:
print(f"{len(X_train)/len(X)*100}%")
print(f"{len(X_val)/len(X)*100}%")

79.99954326429012%
20.000456735709882%


In [6]:
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

In [7]:
y_train = y_train.values.reshape(-1, 1)
y_val = y_val.values.reshape(-1, 1)

In [8]:
X_scaler.fit(X_train)
y_scaler.fit(y_train)

MinMaxScaler()

In [9]:
X_train_scaled = X_scaler.transform(X_train)
X_val_scaled = X_scaler.transform(X_val)

In [10]:
y_train_scaled = y_scaler.transform(y_train)
y_val_scaled = y_scaler.transform(y_val)

## 1. Selección del Algoritmo
Hemos seleccionado **Random Forest** como nuestro algoritmo base por las siguientes razones:
- Excelente rendimiento en problemas de clasificación binaria
- Manejo robusto de datos desbalanceados (común en detección de fraude)
- Capacidad para manejar tanto variables numéricas como categóricas
- Menor tendencia al sobreajuste comparado con árboles individuales
- Proporciona importancia de características, útil para entender factores de fraude

In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, truncnorm, randint
from sklearn.ensemble import RandomForestClassifier
import time

## 2. Configuración de los Modelos
Implementamos tres variantes del Random Forest con diferentes enfoques:

### Modelo 1: Random Forest con Configuración Básica

In [12]:
from sklearn.ensemble import RandomForestClassifier

In [13]:
rf1 = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

In [14]:
y_train_scaled = y_train_scaled.ravel()

In [15]:
rf1.fit(X_train_scaled, y_train_scaled)

RandomForestClassifier(random_state=42)

### Modelo 3: Random Forest con Optimización de Características

In [16]:
rf2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)

In [17]:
rf2.fit(X_train_scaled, y_train_scaled)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=2, min_samples_split=5,
                       n_estimators=200, random_state=42)

**Justificación**: Optimizado para seleccionar características relevantes y reducir ruido en la predicción.

### Modelo 3: Random Forest con Optimización de Características

In [18]:
# Configuration with feature selection
rf3 = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_split=3,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)

In [19]:
rf3.fit(X_train_scaled, y_train_scaled)

RandomForestClassifier(max_depth=15, min_samples_leaf=2, min_samples_split=3,
                       n_estimators=150, random_state=42)

**Justificación**: Optimizado para seleccionar características relevantes y reducir ruido en la predicción.

## Evalúe y compare todos los modelos.
Interprete los resultados y concluya cuál modelo es el mejor. Puede considerar más de una métrica de evaluación para su conclusión.


## 3. Proceso de Evaluación
Implementamos un proceso de evaluación robusto:

In [20]:
from sklearn.metrics import balanced_accuracy_score, classification_report

In [21]:
models = [rf1, rf2, rf3]
results = {}

In [22]:
for i, model in enumerate(models, 1):
    print(f"\nEvaluando Modelo {i}:")
    print("-" * 50)
    
    # Realizar predicciones con datos escalados
    y_pred_scaled = model.predict(X_val_scaled)
    
    # Calcular la precisión balanceada
    balanced_acc = balanced_accuracy_score(y_val, y_pred_scaled)
    
    # Almacenar resultados en el diccionario
    results[f"Model_{i}"] = {
        "balanced_accuracy": balanced_acc,
        "classification_report": classification_report(y_val, y_pred_scaled)
    }
    
    # Imprimir resultados
    print(f"Precisión Balanceada: {balanced_acc:.4f}")
    print("\nInforme de Clasificación:")
    print(classification_report(y_val, y_pred_scaled))

# Encontrar el mejor modelo
best_model_name = max(results, key=lambda k: results[k]["balanced_accuracy"])
print(f"\nMejor Modelo: {best_model_name}")
print(f"Mejor Precisión Balanceada: {results[best_model_name]['balanced_accuracy']:.4f}")


Evaluando Modelo 1:
--------------------------------------------------
Precisión Balanceada: 0.7016

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     34120
           1       0.98      0.40      0.57       912

    accuracy                           0.98     35032
   macro avg       0.98      0.70      0.78     35032
weighted avg       0.98      0.98      0.98     35032


Evaluando Modelo 2:
--------------------------------------------------
Precisión Balanceada: 0.7094

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99     34120
           1       0.51      0.43      0.47       912

    accuracy                           0.97     35032
   macro avg       0.75      0.71      0.73     35032
weighted avg       0.97      0.97      0.97     35032


Evaluando Modelo 3:
--------------------------------------------------
Precisión Balancea

Encontramos el mejor modelo

In [23]:
best_model_name = max(results, key=lambda k: results[k]["balanced_accuracy"])
print(f"\nBest Model: {best_model_name}")
print(f"Best Balanced Accuracy: {results[best_model_name]['balanced_accuracy']:.4f}")


Best Model: Model_2
Best Balanced Accuracy: 0.7094


In [26]:
df_test = pd.read_csv(f"{converted_dir}/vehiculos_test.csv")

X_test = df_test[predictor_vars].reset_index(drop=True)

In [28]:
X_test

,anio_registro,millas_recorridas,precio_vehiculo,num_asientos,num_puertas,complejidad_reparacion,costo_reparacion,horas_reparacion,marca2_Audi,marca2_BMW,marca2_Citroen,marca2_Ford,marca2_Nissan,marca2_OTROS,marca2_Peugeot,marca2_Toyota,marca2_Vauxhall,marca2_Volkswagen
0,2015.0,37723.0,12690.0,5.0,4.0,1,23.807,1.0,0,0,0,0,0,0,0,0,0,1
1,2014.0,78000.0,13490.0,5.0,4.0,3,73.490,3.0,0,0,0,0,0,1,0,0,0,0
2,2010.0,44936.0,8995.0,5.0,5.0,1,160.000,2.0,0,0,0,0,0,0,0,0,0,1
3,2014.0,29000.0,22990.0,4.0,3.0,1,42.990,1.0,1,0,0,0,0,0,0,0,0,0
4,2016.0,18037.0,9293.0,5.0,5.0,1,28.586,0.5,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19454,2014.0,43000.0,9795.0,7.0,5.0,1,25.877,1.0,0,0,0,0,0,0,1,0,0,0
19455,2013.0,39351.0,7990.0,5.0,5.0,1,43.995,2.0,0,0,0,0,0,0,0,1,0,0
19456,2009.0,100000.0,1800.0,5.0,5.0,1,40.900,2.0,0,0,0,0,0,1,0,0,0,0
19457,2012.0,69000.0,3995.0,5.0,5.0,1,119.900,2.0,0,0,0,0,0,0,0,0,1,0


In [27]:
y_test_pred = models[best_model_name].predict(X_test)

TypeError: list indices must be integers or slices, not str

In [ ]:
import os

os.makedirs(predicted_dir,exist_ok=True)

In [ ]:
y_test_pred.to_csv(f"{predicted_dir}/vehiculos_test_preds.csv")